In [1]:
!pip uninstall -y massi
!pip cache purge

Found existing installation: massi 0.1.0
Uninstalling massi-0.1.0:
  Successfully uninstalled massi-0.1.0
Files removed: 6 (1.2 MB)


In [2]:
!cd .. && pip install -v . --group=dev --group=observability --group=sandbox

Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
Processing /home/jovyan/work
  Running command installing build dependencies
  Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
    Obtaining dependency information for setuptools>=40.8.0 from https://files.pythonhosted.org/packages/9d/76/f789f7a86709c6b087c5a2f52f911838cad707cc613162401badc665acfe/setuptools-82.0.1-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.0 MB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━ 0.8/1.0 MB 7.9 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 4.0 MB/s  0:00:00
  Installing build dependencies ... done
  Running command Getting requirements to build wheel
  running egg_info
  writing massi.egg-info/PKG-INFO
  writing dependency_links to massi.egg-info/dependency_links.txt
  writing requirements to massi.egg-info/requires.txt
  writing top-level names to massi.egg-info/top_level.txt

In [3]:
# Setup observability
USE_OBSERVABILITY = True

if USE_OBSERVABILITY:
    import os
    from lmnr import Laminar
    
    Laminar.initialize(
        project_api_key=os.environ["LMNR_PROJECT_API_KEY"],
        base_url=f"http://{os.environ["HOST_MACHINE_URL_CORE"]}",
        http_port=int(os.environ["LMNR_PROJECT_HTTP_PORT"]),
        grpc_port=int(os.environ["LMNR_PROJECT_GRPC_PORT"]),
    )

In [4]:
from massi.agents.massi import MassiScenarioFactory, Config, Dependencies, StandardIOPromptClient

In [5]:
from massi.runtimes.jupyter.ui import JupyterTextWidgetIOPromptClient

In [6]:
from massi.agents.massi import create_ollama_llm

base_config = Config(**{"model_url": "http://host.docker.internal:11434", "model_name": "granite4:3b"})
base_deps = Dependencies(llm_client=create_ollama_llm(base_config), prompt_client=JupyterTextWidgetIOPromptClient())

In [7]:
import random

thread = {"configurable": {"thread_id": str(random.randint(1, 100))}}

In [8]:
import pprint

In [9]:
from massi.runtimes.flow import stream_until_iterruption

In [10]:
from massi.agents.constructs import GroupedGisResourceFile

graph_invocation_input_state = {
    "resource":  GroupedGisResourceFile(
        file_location='data/layer_data/Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3.csv', 
        filename='Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3.csv', 
        extension='.csv', 
        author='jovyan', 
        gis_relevancy_score=1.0, 
        resource_name='Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3', 
        group_score=1.0,
    )
}

In [11]:
from massi.agents.constructs import QueryStep

In [12]:
query_step = QueryStep(**{
    'id': 1, 
    'key': 'location', 
    'json_path': '$.location', 
    'priority': 5, 
    'query': 'Please provide the primary location of this digital resource, such as a file path or URI.'
})

# 2. Data Explorer Agent

In [13]:
from massi.agents.crafters.data_explorer import build_data_explorer_graph

In [14]:
data_explorer_graph = build_data_explorer_graph(base_deps)

In [15]:
from massi.agents.scenarios.resource_properties import Defaults

In [16]:
data_explorer_graph.invoke({
    **graph_invocation_input_state, 
    "query": query_step, 
    "context": {"target_schema": Defaults.resource_properties_metadata_json_schema}
})["extraction_result"]

"The 'location' key is not present in the provided dataframe. Therefore, there is no value to extract for 'location'."